## Setup
- Imports
- Define Helper Function. 
- Define and show relevent paths
- Get FFMPEG Dependency by Disecting a bundled package (docker eat your heartout).

In [20]:
import os
import pandas as pd
import shutil
import tempfile
import subprocess
from pathlib import Path
import imageio_ffmpeg
import whisper
import whisper.audio
import time
import re

In [21]:
def extract_title(filename, *, safe_filename=False):
    """Extract a recording title; optionally make it safe for use as a filename."""
    name = os.path.basename(os.fspath(filename))
    name = re.sub(r"\.(?:opus|mp3)$", "", name, flags=re.IGNORECASE) # Remove file extension
    name = re.sub(r"^\d+\s*-\s*", "", name).strip() # Remove leading numbers and dashes, and strip whitespace

    if safe_filename:
        name = re.sub(r'[<>:"/\\|?*]', "_", name).strip(" .") # Replace invalid filename characters with underscores and strip leading/trailing spaces and dots

    return name

In [ ]:
# Configure Whisper to use imageio-ffmpeg directly (Docker Eat your heart out!)
ffmpeg_path = imageio_ffmpeg.get_ffmpeg_exe()
whisper.audio.FFMPEG = ffmpeg_path

print("FFmpeg version:", imageio_ffmpeg.get_ffmpeg_version())

BASE_FOLDER = Path(DOWNLOAD_FOLDER)
TRANSCRIPT_FOLDER = Path("../../Data/Processed/transcripts")
TRANSCRIPT_FOLDER.mkdir(parents=True, exist_ok=True)

print("DOWNLOAD_FOLDER:", os.path.abspath(DOWNLOAD_FOLDER))
print("Base folder:", BASE_FOLDER)
print("Transcript folder:", TRANSCRIPT_FOLDER)
print("Number of audio files found:", len(list(BASE_FOLDER.glob("*.opus"))))

FFmpeg version: 9.0.2-essentials_build-www.gyan.dev
DOWNLOAD_FOLDER: c:\Users\Graem\Desktop\Github Repos\EMF_Camp_DataEng\Data\Landing_Zone\Recordings
Base folder: ..\..\Data\Landing_Zone\Recordings
Transcript folder: ..\..\Data\Landing_Zone\Recordings\transcript
Number of audio files found: 130


In [19]:
# Configure Whisper to use imageio-ffmpeg directly (Docker Eat your heart out!)
ffmpeg_path = imageio_ffmpeg.get_ffmpeg_exe()
whisper.audio.FFMPEG = ffmpeg_path

print("FFmpeg version:", imageio_ffmpeg.get_ffmpeg_version())

# Define folders
BASE_FOLDER = Path(DOWNLOAD_FOLDER)
TRANSCRIPT_FOLDER = BASE_FOLDER / "transcript"
TRANSCRIPT_FOLDER.mkdir(parents=True, exist_ok=True)

# Get original audio filenames
audio_files = sorted([
    filename
    for filename in BASE_FOLDER.iterdir()
    if filename.is_file()
    and filename.suffix.lower() in (".mp3", ".opus") # included other formats to make obvious how to add more formats if needed
])

total_files = len(audio_files)
print(f"\nFound {total_files} recordings.")

# Choose Whisper model
MODEL_NAME = "tiny"
print(f"Loading Whisper model: {MODEL_NAME}")
model = whisper.load_model(MODEL_NAME)

# Start transcription process
if total_files == 0:
    print("No audio files found. Check your download folder.")

else:
    start_time = time.time()
    processed_times = []

    # Transcribe each audio file
    for index, audio_file in enumerate(audio_files, 1):

        transcript_title = extract_title(
            audio_file.name,
            safe_filename=True
        )

        output_file = TRANSCRIPT_FOLDER / f"{transcript_title}.txt"

        # Skip files already transcribed
        if output_file.exists():
            print(f"[{index}/{total_files}] Skipping: {audio_file.name}")
            continue

        print(f"\n[{index}/{total_files}] Transcribing: {audio_file.name}")
        file_start = time.time()

        try:
            # Transcribe using Whisper and the configured FFmpeg executable
            result = model.transcribe(
                str(audio_file),
                language="en",
                fp16=False,
                verbose=False,
            )

            # Save timestamped transcript
            with output_file.open("w", encoding="utf-8") as f:
                for segment in result["segments"]:
                    start = segment["start"]
                    text = segment["text"].strip()

                    minutes = int(start // 60)
                    seconds = int(start % 60)

                    f.write(f"[{minutes:02d}:{seconds:02d}] {text}\n")

            # Calculate processing time
            elapsed = time.time() - file_start
            processed_times.append(elapsed)

            # Estimate remaining time
            remaining_files = sum(
                1 for remaining_file in audio_files[index:]
                if not (
                    TRANSCRIPT_FOLDER /
                    f"{extract_title(remaining_file.name, safe_filename=True)}.txt"
                ).exists()
            )

            average_time = sum(processed_times) / len(processed_times)
            remaining_seconds = remaining_files * average_time
            minutes, seconds = divmod(int(remaining_seconds), 60)

            print(f"Saved: {output_file.name}")
            print(
                f"Progress: {index}/{total_files} | "
                f"Elapsed: {(time.time() - start_time)/60:.1f} min | "
                f"ETA: {minutes}m {seconds}s"
            )

        except Exception as error:
            print(f"ERROR processing {audio_file.name}: {error}")

    print("\nAll transcription jobs completed.")

FFmpeg version: 9.0.2-essentials_build-www.gyan.dev

Found 130 recordings.
Loading Whisper model: tiny

[1/130] Transcribing: 001 - Men in tech #YesAllMen.opus


 10%|█         | 18600/179886 [00:13<02:01, 1332.31frames/s]


KeyboardInterrupt: 

# Temp below


In [ ]:
import os
import shutil
import tempfile
import subprocess
from pathlib import Path
import imageio_ffmpeg
import whisper

# Locate the FFmpeg executable
ffmpeg_source = Path(imageio_ffmpeg.get_ffmpeg_exe())

# Remember the original PATH
original_path = os.environ.get("PATH")

try:
    # Create a temporary directory (automatically deleted afterwards)
    with tempfile.TemporaryDirectory(prefix="emf_camp_ffmpeg_") as temp_dir:

        # Copy FFmpeg into the temporary directory
        ffmpeg_alias = Path(temp_dir) / "ffmpeg.exe"
        shutil.copy2(ffmpeg_source, ffmpeg_alias)

        # Temporarily make FFmpeg available to Whisper
        os.environ["PATH"] = (
            str(temp_dir) + os.pathsep + (original_path or "")
        )

        # Verify FFmpeg works
        subprocess.run(
            ["ffmpeg", "-version"],
            check=True,
            capture_output=True
        )

        print("FFmpeg configured successfully!")

        # Get the audio file path
        audio_path = Path(audio_files[0])

        # If audio_files contains filenames rather than full paths
        if not audio_path.is_absolute():
            audio_path = Path(DOWNLOAD_FOLDER) / audio_path

        # Load audio using Whisper
        audio = whisper.load_audio(str(audio_path))

        print("Audio loaded successfully!")
        print("Audio shape:", audio.shape)

finally:
    # Restore the original PATH, even if an error occurs
    if original_path is None:
        os.environ.pop("PATH", None)
    else:
        os.environ["PATH"] = original_path

print("Temporary FFmpeg files cleaned up!")

In [ ]:
import os
import shutil
import tempfile
import subprocess
from pathlib import Path
import imageio_ffmpeg
import whisper

# Locate the FFmpeg executable
ffmpeg_source = Path(imageio_ffmpeg.get_ffmpeg_exe())

# Remember the original PATH
original_path = os.environ.get("PATH")

try:
    # Create a temporary directory (automatically deleted afterwards)
    with tempfile.TemporaryDirectory(prefix="emf_camp_ffmpeg_") as temp_dir:

        # Copy FFmpeg into the temporary directory
        ffmpeg_alias = Path(temp_dir) / "ffmpeg.exe"
        shutil.copy2(ffmpeg_source, ffmpeg_alias) # Creates a copy of FFmpeg named ffmpeg.exe in a temporary folder so that Whisper can find and execute it.

        # Temporarily make FFmpeg available to Whisper
        os.environ["PATH"] = (
            str(temp_dir) + os.pathsep + (original_path or "")
        )

        # Verify FFmpeg works
        subprocess.run(
            ["ffmpeg", "-version"],
            check=True,
            capture_output=True
        )

        print("FFmpeg configured successfully!")

        # Get the audio file path
        audio_path = Path(audio_files[0])

        # If audio_files contains filenames rather than full paths
        if not audio_path.is_absolute():
            audio_path = Path(DOWNLOAD_FOLDER) / audio_path

        # Load audio using Whisper
        audio = whisper.load_audio(str(audio_path))

        print("Audio loaded successfully!")
        print("Audio shape:", audio.shape)

finally:
    # Restore the original PATH, even if an error occurs
    if original_path is None:
        os.environ.pop("PATH", None)
    else:
        os.environ["PATH"] = original_path

print("Temporary FFmpeg files cleaned up!")

In [ ]:
# Locate the bundled FFmpeg executable
ffmpeg_source = Path(imageio_ffmpeg.get_ffmpeg_exe())

# Create a temporary folder containing an executable named ffmpeg.exe
ffmpeg_folder = Path(tempfile.gettempdir()) / "emf_camp_ffmpeg"
ffmpeg_folder.mkdir(exist_ok=True)

ffmpeg_alias = ffmpeg_folder / "ffmpeg.exe"
shutil.copy2(ffmpeg_source, ffmpeg_alias) # : Creates a copy of FFmpeg named ffmpeg.exe in a temporary folder so that Whisper can find and execute it.

# Add the folder to PATH so Whisper can find FFmpeg
os.environ["PATH"] = (
    str(ffmpeg_folder) + os.pathsep + os.environ.get("PATH", "")
)

# Test FFmpeg
subprocess.run(
    ["ffmpeg", "-version"],
    check=True,
    capture_output=True
)

print("FFmpeg configured successfully!")

# Test Whisper audio loading
audio_path = os.path.join(DOWNLOAD_FOLDER, audio_files[0])

# Check that the audio file exists
assert os.path.exists(audio_path), f"Audio file not found: {audio_path}"

audio = whisper.load_audio(audio_path)

print("Audio loaded successfully!")
print("Audio shape:", audio.shape)